# Génération d'image & jeu avec le gradient

L'idée est de récupérer une architecture pré-entrainée et d'aller jouer avec. Deux options assez simples sont envisageables:
1. Les approches *deepdream* : [github](https://github.com/google/deepdream/tree/master); [LeMonde](https://www.lemonde.fr/pixels/article/2015/07/09/on-a-teste-pour-vous-deep-dream-la-machine-a-reves-psychedeliques-de-google_4675562_4408996.html)
1. Les approches *style/content*, dites aussi transfert de style [github](https://github.com/Raed-Alshehri/Art_Generation_Neural_Style_Transfer). Si j'ai une image A qui m'interesse pour son contenu et une image B (i.e. un Van Gogh ou Munch) qui m'interesse pour son style, suis-je capable de construire une version A' qui prend le style de B?

Dans tous les cas, il faut récupérer un modèle (par exemple GoogLeNet) pré-entrainé puis récupérer et contraindre des couches intermédiaires (=définir une loss) puis ensuite activer le gradient sur l'entrée et laisser dériver l'image

**Plan du notebook**

0. Mise en place : librairies, *device*, chargement du modèle pré-entraîné
1. A. Classification d'image (vérifier que le modèle fonctionne)
1. B. Les *hooks* : espionner (et modifier) l'intérieur d'un réseau
1. C. Deepdream
1. D. Reconstruction d'image à partir d'un embedding
1. E. Transfert de style (version naïve)
1. F. Transfert de style « à la Gatys » : la matrice de Gram

## 0. Mise en place


In [ ]:
import os
import copy
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.nn import functional as F
from torch.optim.lr_scheduler import OneCycleLR  # pour accélérer l'optimisation (parties D & E)
import torchvision
from torchvision import models, transforms

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("torch :", torch.__version__, "| torchvision :", torchvision.__version__)
print("device :", device)

### Chargement du modèle pré-entraîné (API `weights` de torchvision)

Depuis torchvision 0.13, on ne passe plus par `torch.hub.load(...)` ni par `pretrained=True` (déprécié) : on choisit explicitement un jeu de **poids** (`GoogLeNet_Weights.IMAGENET1K_V1`, ou `.DEFAULT` = les meilleurs poids disponibles).

L'objet `weights` embarque aussi les **méta-données** utiles :
- `weights.transforms()` : le pré-traitement exact utilisé à l'entraînement (resize, crop, normalisation)
- `weights.meta["categories"]` : les 1000 étiquettes ImageNet (plus besoin de télécharger un fichier de labels)

> Remarque : on pourrait aussi récupérer des modèles sur HuggingFace (via `timm` ou `transformers`), avec un catalogue beaucoup plus large (ConvNeXt, ViT, ...). Pour ce TP, torchvision suffit largement : pas de dépendance supplémentaire, et les couches de GoogLeNet ont des noms explicites (`inception3a`, `inception4b`...) très pratiques pour poser des *hooks*.

In [ ]:
from torchvision.models import GoogLeNet_Weights

weights = GoogLeNet_Weights.IMAGENET1K_V1      # ou GoogLeNet_Weights.DEFAULT
model = models.googlenet(weights=weights).to(device)
model.eval()          # mode inférence : dropout désactivé, batchnorm figée

# On n'optimisera jamais les poids du réseau, uniquement l'image d'entrée
# => inutile de calculer/stocker leurs gradients
for p in model.parameters():
    p.requires_grad_(False)

preprocess = weights.transforms()              # pré-traitement associé aux poids
categories = weights.meta["categories"]        # étiquettes ImageNet
MEAN, STD  = preprocess.mean, preprocess.std   # normalisation (réutilisée plus loin)

print(preprocess)
print(len(categories), "classes, par ex.", categories[:5])

In [ ]:
# Pour information : le catalogue des modèles de classification disponibles dans torchvision
print(models.list_models(module=models))

## A. Classification d'image

Vérifier que le réseau marche bien en classification d'image

1. Charger une image
1. Appliquer le classifieur
1. Lire la signification des étiquettes

In [ ]:
# Télécharger une image d'exemple (si elle n'est pas déjà là)
url, filename = ("https://github.com/pytorch/hub/raw/master/images/dog.jpg", "img/dog.jpg")
if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

In [ ]:
input_image = Image.open(filename).convert("RGB")
input_batch = preprocess(input_image).unsqueeze(0).to(device)  # mini-batch de taille 1

with torch.no_grad():
    output = model(input_batch)
# Tensor of shape 1000, with confidence scores over ImageNet's 1000 classes
print(output[0])
# The output has unnormalized scores. To get probabilities, you can run a softmax on it.
probabilities = F.softmax(output[0], dim=0)
print(probabilities)

In [ ]:
# Show top categories per image
top5_prob, top5_catid = torch.topk(probabilities, 5)
for i in range(top5_prob.size(0)):
    print(categories[top5_catid[i]], top5_prob[i].item())

In [ ]:
plt.imshow(input_image)
plt.axis("off")
plt.show()

## B. Les *hooks* : espionner (et modifier) l'intérieur d'un réseau

### B.0 Pourquoi ?

Quand on appelle `model(x)`, PyTorch ne renvoie que la **sortie finale**. Or pour la suite (deepdream, reconstruction, transfert de style), on a besoin des **activations intermédiaires** (sortie de `inception3a`, `inception4b`...).

Plusieurs options :
1. réécrire le `forward` du modèle → pénible, et on ne veut pas toucher au code d'un modèle pré-entraîné
1. découper le modèle (`nn.Sequential(*list(model.children())[:k])`) → fragile dès que l'architecture n'est pas une simple séquence (branches parallèles, connexions résiduelles...)
1. **poser un crochet (*hook*)** : une fonction que PyTorch appellera automatiquement à chaque fois qu'un module donné est exécuté.

**Analogie** : un hook, c'est une sonde que l'on branche sur un câble du circuit. Le courant continue de passer normalement, mais on peut lire (voire modifier) ce qui y circule.

| Finalité | Exemples |
|---|---|
| Observer | visualiser des cartes d'activation, déboguer (formes, NaN...) |
| Extraire | récupérer des *features* intermédiaires (transfer learning, similarité, *perceptual loss*) |
| Optimiser | définir une loss sur une couche cachée : **deepdream, reconstruction, transfert de style** (parties C, D, E) |
| Modifier | remplacer/masquer une activation à la volée (ablation, *pruning*...) |

Le cycle de vie d'un hook se fait en 3 temps :
1. **poser** : `handle = module.register_forward_hook(fonction)`
1. **utiliser** : chaque `model(x)` déclenche `fonction(module, input, output)`
1. **retirer** : `handle.remove()`

### B.1 Un premier hook sur un réseau jouet

On commence avec un minuscule réseau pour bien voir ce qui se passe.

In [ ]:
torch.manual_seed(0)
toy = nn.Sequential(
    nn.Linear(4, 3),   # toy[0]
    nn.ReLU(),         # toy[1]
    nn.Linear(3, 2),   # toy[2]
)
x = torch.randn(1, 4)

print(">>> forward initial")
y = toy(x)                                         # 0. froward classique
print("sortie finale :", y)

# La signature est imposée par PyTorch : (module, input, output)
#  - module : le module sur lequel le hook est posé
#  - input  : un TUPLE contenant les entrées du module (un module peut avoir plusieurs entrées)
#  - output : la sortie du module
def print_hook(module, input, output):
    print(f"  [hook] {module.__class__.__name__} : entrée {tuple(input[0].shape)} -> sortie {tuple(output.shape)}")
    print("         sortie =", output)

handle = toy[1].register_forward_hook(print_hook)   # 1. poser

print(">>> forward avec le hook")
y = toy(x)                                         # 2. le hook se déclenche PENDANT le forward
print("sortie finale :", y)

handle.remove()                                    # 3. retirer

print("\n>>> forward après remove : le hook ne s'affiche plus")
y = toy(x)
print("sortie finale :", y)

À retenir : le hook n'a **rien changé** au calcul (même sortie finale), il a seulement observé ce qui passait dans la ReLU.

### B.2 Stocker les activations dans un dictionnaire

Afficher c'est bien, mais on veut surtout **récupérer** les valeurs pour s'en servir ensuite. On ne peut pas récupérer un résultat « renvoyé » par le hook (un `return` a un autre sens, cf. B.5) : on écrit donc dans une structure extérieure, typiquement un **dictionnaire**.

Pour poser le même type de hook sur plusieurs couches, on utilise une **fabrique de hooks** : une fonction qui *renvoie* une fonction hook, en mémorisant le nom de la couche.

In [ ]:
activations = {}

def save_activation(name):
    # Fabrique un hook qui range la sortie du module dans activations[name]
    def hook(module, input, output):
        activations[name] = output
    return hook

# parcours de la structure Sequential
handles = [toy[i].register_forward_hook(save_activation(f"couche{i}")) for i in range(3)]
y = toy(x)
for h in handles: # tjs penser à les enlever (cf plus loin)
    h.remove()

for name, act in activations.items():
    print(name, tuple(act.shape), act)

### B.3 Sur un vrai réseau : trouver les couches puis y accrocher des sondes

Dans GoogLeNet, chaque sous-module a un nom (`conv1`, `inception3a`, `inception4b.branch2`, ...) :
- `model.named_children()` liste le premier niveau,
- `model.named_modules()` liste tout, récursivement,
- `model.get_submodule("inception4b")` récupère un module à partir de son nom.

In [ ]:
for name, module in model.named_children():
    print(f"{name:15s} {module.__class__.__name__}")

In [ ]:
layers = ["conv1", "inception3a", "inception3b", "inception4a", "inception4e", "inception5b"]

activations = {}
handles = [model.get_submodule(name).register_forward_hook(save_activation(name)) for name in layers]

with torch.no_grad():
    model(input_batch)        # l'image du chien, préparée en partie A

for h in handles:
    h.remove()

for name in layers:
    print(f"{name:12s} -> {tuple(activations[name].shape)}")

On observe que le nombre de canaux augmente et que la résolution spatiale diminue au fil des couches. Les couches basses encodent des motifs locaux (bords, textures), les couches profondes des concepts plus abstraits : c'est exactement ce que nous allons exploiter en C, D et E.

> Pour la seule *extraction* de features, torchvision propose aussi `torchvision.models.feature_extraction.create_feature_extractor(model, return_nodes=[...])`, qui fait la même chose sans hook explicite. Les hooks restent plus généraux (on peut aussi modifier les activations, ou les poser sur n'importe quel modèle).

### B.4 Exercice : visualiser des cartes d'activation

Afficher les 8 premières cartes d'activation (canaux) de `inception3a` et de `inception4e` pour l'image du chien.

*Note* : la carte du canal `c` est `activations[name][0, c]`, à ramener sur le CPU (`.cpu()`) pour matplotlib.

*Note 2*: il est clair que certaines activations sont plus proches du contenu et d'autres plus abstraites

In [ ]:

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for row, name in enumerate(["inception3a", "inception4e"]):
    act = activations[name][0].cpu()
    for c in range(8):
        axes[row, c].imshow(act[c], cmap="viridis")
        axes[row, c].set_title(f"{name}[{c}]", fontsize=9)
        axes[row, c].axis("off")
plt.tight_layout()
plt.show()


### B.5 Un hook peut modifier la sortie

Si la fonction hook **renvoie** une valeur (autre que `None`), celle-ci **remplace** la sortie du module pour la suite du forward. Exemple : on « éteint » la ReLU du réseau jouet.

C'est une fonctionnalité critique pour implémenter des ablations et comprendre l'origine de la performance dans les architectures neuronales

In [ ]:
def kill_hook(module, input, output):
    return torch.zeros_like(output)      # la sortie de la ReLU est remplacée par des 0

print("sans hook :", toy(x))
handle = toy[1].register_forward_hook(kill_hook)
print("avec hook :", toy(x), " <- il ne reste que le biais de la dernière couche")
print("biais     :", toy[2].bias)
handle.remove()

Même principe sur GoogLeNet : on éteint la moitié des canaux du bloc `inception4c` (*ablation*) et on regarde l'effet sur la prédiction.

In [ ]:
def top5(batch):
    with torch.no_grad():
        p = F.softmax(model(batch)[0], dim=0)
    vals, ids = torch.topk(p, 5)
    return [(categories[i], round(v.item(), 3)) for v, i in zip(vals, ids)]

def ablation_hook(module, input, output):
    out = output.clone()
    out[:, : out.shape[1] // 2] = 0      # on met à zéro la première moitié des canaux
    return out

print("normal   :", top5(input_batch))
handle = model.get_submodule("inception4c").register_forward_hook(ablation_hook)
print("ablation :", top5(input_batch))
handle.remove()

### B.6 Hooks et gradient : la clé pour la suite

Point essentiel : le tenseur stocké par le hook **fait partie du graphe de calcul**. Si l'entrée `x` a `requires_grad=True`, on peut définir une loss sur `activations[...]` puis faire `loss.backward()` : le gradient remonte jusqu'à `x`.

C'est tout le principe des parties C, D et E : on optimise **l'image**, pas les poids du réseau.

(À l'inverse, si on veut seulement *stocker* des features sans garder le graphe — pour économiser la mémoire — on range `output.detach()`.)

In [ ]:
x = torch.randn(1, 4, requires_grad=True)

activations = {}
handle = toy[0].register_forward_hook(save_activation("couche0"))
toy(x)                                  # le forward remplit activations["couche0"]
handle.remove()

loss = activations["couche0"].norm()    # une loss définie sur une couche CACHÉE
loss.backward()
print("gradient de la loss par rapport à l'entrée x :", x.grad)

### B.7 Piège classique : oublier `remove()`

Chaque appel à `register_forward_hook` **ajoute** un hook, il ne remplace pas le précédent. Si on ré-exécute une cellule sans avoir retiré le hook, les hooks s'accumulent : calculs en double, mémoire qui gonfle, résultats incompréhensibles...

In [ ]:
counter = {"appels": 0}
def count_hook(module, input, output):
    counter["appels"] += 1

for _ in range(3):                             # on simule 3 exécutions de la même cellule...
    toy[1].register_forward_hook(count_hook)   # ... sans conserver le handle !

toy(x)
print("nombre d'appels du hook pour UN SEUL forward :", counter["appels"])
print("hooks présents sur toy[1] :", len(toy[1]._forward_hooks))

toy[1]._forward_hooks.clear()   # remède de secours (attribut privé) : effacer tous les hooks du module

**Bonnes pratiques**
- toujours conserver le `handle` et appeler `handle.remove()` dès que le hook n'est plus utile ;
- se mettre en `model.eval()` quand on observe un modèle pré-entraîné ;
- il existe d'autres hooks : `register_forward_pre_hook` (avant le forward, reçoit l'entrée), `register_full_backward_hook` (pendant le backward, reçoit les gradients), `tensor.register_hook` (sur le gradient d'un tenseur). Nous n'en aurons pas besoin ici.

## C. Deepdream

1. Charger l'image + activer le gradient
1. La passer dans le réseau et récupérer une couche cachée (avec un hook, cf. B)
1. Définir une loss sur cette couche
1. Rétro-propager l'erreur

In [ ]:
# --- Charger une image PIL et la prétraiter (sans crop, pour garder toute l'image) ---
def load_image(path, max_size=500):
    image = Image.open(path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize(max_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])
    return transform(image).unsqueeze(0).to(device)

# --- Opération inverse : tenseur normalisé -> image PIL affichable ---
def deprocess(tensor):
    tensor = tensor.detach().cpu().squeeze(0)
    tensor = tensor * torch.tensor(STD).view(3, 1, 1)
    tensor = tensor + torch.tensor(MEAN).view(3, 1, 1)
    tensor = torch.clamp(tensor, 0, 1)
    return transforms.ToPILImage()(tensor)


img = load_image("img/dog.jpg")
img_orig = img.clone()

On réutilise la fabrique de hooks `save_activation` de la partie B :
1. On *pose* le crochet sur la couche choisie
1. On fait le forward
1. On a l'information dans le dictionnaire `activations` : on peut créer notre loss !

In [ ]:
# --- Choisir une couche cible ---
layer_name = 'inception4b'  # peut être 'inception3a', 'inception4d', 'inception4c' etc.

# --- Poser le hook ---
activations = {}
hook = model.get_submodule(layer_name).register_forward_hook(save_activation(layer_name))

# --- DeepDream : ascension de gradient sur l'image ---
img.requires_grad_(True)
optimizer = torch.optim.Adam([img], lr=0.1) # jouer avec le learning rate + nb itération

for i in range(30):  # nombre d'itérations
    optimizer.zero_grad()
    model(img)
    loss = -activations[layer_name].norm()  # maximise la norme des activations ATTENTION à bien mettre un moins !
    loss.backward()
    optimizer.step()

    # Optionnel : normaliser légèrement les pixels pour la stabilité
    img.data = torch.clamp(img.data, -2, 2)

    if i % 5 == 0:
        print(f"Iteration {i}, Loss = {loss.item():.2f}")

hook.remove()

In [ ]:
# --- Convertir l'image en format affichable ---
dream_img = deprocess(img)
img_orig2  = deprocess(img_orig)
dream_img_c = dream_img.crop((50, 50, 200, 200))
img_orig_c = img_orig2.crop((50, 50, 200, 200))


plt.figure(figsize=(20,5))

plt.subplot(1,4,1)
plt.imshow(img_orig2)
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(dream_img)
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(img_orig_c)
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(dream_img_c)
plt.axis("off")


plt.show()

## D. Reconstruction d'image

1. A *encoder* (=forward) une image dans un réseau
1. Isoler une représentation de cette image... Permettra-t-elle de reconstruire l'image?
1. Partir d'une image aléatoire (bruit blanc gaussien) avec gradient activé
1. Minimiser la distance au contenu encodé dans l'embedding

On va travailler avec le même réseau. Mais on introduit différentes optimisation pour aller plus vite en calcul (`OneCycleLR`).

Le transfert de style sera à faire par vous même... Mais afin de simplifier les choses, je vous propose un code permettant la reconstruction d'une image à partir d'un embedding

In [ ]:
img = load_image("img/dog.jpg")

layer_content = 'inception3a'  # la première (bon encodage du contenu)
activations = {}
hook = model.get_submodule(layer_content).register_forward_hook(save_activation(layer_content))


# target = une couche de l'encodage d'une image
with torch.no_grad():
    model(img)
    target = activations[layer_content].clone()
    print("dimension de l'embedding : ", target.shape)


# Générer une image aléatoire
noise = torch.randn_like(img) #*0.1

# --- Generation d'image : minimiser la norme entre les embeddings ---
noise.requires_grad_(True)

nepoch = 100 # on peut aller à 1000 sur les machines puissantes
max_lr=5e-1 # version optimisée OneCycle
optimizer = torch.optim.AdamW([noise], lr=max_lr)
scheduler = OneCycleLR(optimizer, max_lr=max_lr, steps_per_epoch=1, epochs=nepoch)

# optimizer = torch.optim.Adam([noise], lr=0.05) # jouer avec le learning rate + nb itération

for i in range(nepoch):  # nombre d'itérations
    optimizer.zero_grad()
    model(noise)
    loss = (target-activations[layer_content]).norm()
    loss.backward()
    optimizer.step()
    scheduler.step()

    # Optionnel : normaliser légèrement les pixels pour la stabilité
    noise.data = torch.clamp(noise.data, -2, 2)

    if i % 5 == 0:
        print(f"Iteration {i}, Loss = {loss.item():.2f}")

    # if i % 200 == 0:  # Si vous voulez voir la progression
    #     os.makedirs("save", exist_ok=True)
    #     deprocess(noise).save("save/image"+str(i)+".jpg")

hook.remove()

In [ ]:
# --- Convertir l'image en format affichable ---

dream_img = deprocess(noise)
img_orig2  = deprocess(img)
dream_img_c = dream_img.crop((50, 50, 200, 200))
img_orig_c = img_orig2.crop((50, 50, 200, 200))


plt.figure(figsize=(20,5))

plt.subplot(1,4,1)
plt.imshow(img_orig2)
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(dream_img)
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(img_orig_c)
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(dream_img_c)
plt.axis("off")


plt.show()

## E. Transfert de style

Le transfert de style [lien](https://github.com/Raed-Alshehri/Art_Generation_Neural_Style_Transfer) consiste:
1. A *encoder* (=forward) deux images dans un réseau
1. Isoler deux représentations à deux étages différents (couche intermédiaire pour le contenu, couche profonde pour le style)
1. Partir d'une image aléatoire (bruit blanc gaussien) avec gradient activé
1. Minimiser la distance à la fois au contenu et au style (= 2 normes à pondérer)

On va travailler avec le même réseau

A vous de jouer pour le transfert

> Cette première version est volontairement naïve : on verra ses limites, et comment les corriger, dans la partie F.

In [ ]:
img   = load_image("img/dog.jpg")
style = load_image("img/vg.jpg")

In [ ]:
# --- Choisir les couches cibles ---
layer_content = 'inception3a'  # la première
layer_style   = 'inception5a'  # la dernière

### <CORRECTION>

# --- Poser les hooks (un par couche, cf. B.2) ---
activations = {}
hook  = model.get_submodule(layer_content).register_forward_hook(save_activation(layer_content))
hook2 = model.get_submodule(layer_style).register_forward_hook(save_activation(layer_style))

# targets = encodage du contenu (image) et du style
with torch.no_grad():
    model(img)
    target = activations[layer_content].clone()
    model(style)
    target_s = activations[layer_style].clone()

# Générer une image aléatoire
noise = torch.randn_like(img) #*0.1

# --- Optimisation de l'image ---
noise.requires_grad_(True)

nepoch = 2000
max_lr=5e-1
alpha = 0.4
optimizer = torch.optim.AdamW([noise], lr=max_lr)
scheduler = OneCycleLR(optimizer, max_lr=max_lr, steps_per_epoch=1, epochs=nepoch)

# optimizer = torch.optim.Adam([noise], lr=0.05) # jouer avec le learning rate + nb itération

os.makedirs("save", exist_ok=True)
for i in range(nepoch):  # nombre d'itérations
    optimizer.zero_grad()
    model(noise)
    loss = (target-activations[layer_content]).norm()
    loss2 = (target_s-activations[layer_style]).norm()
    comb = (alpha*loss + (1-alpha)*loss2)
    comb.backward()
    optimizer.step()
    scheduler.step()

    # Optionnel : normaliser légèrement les pixels pour la stabilité
    noise.data = torch.clamp(noise.data, -2, 2)

    if i % 5 == 0:
        print(f"Iteration {i}, Loss = {comb.item():.2f}")

    if i % 200 == 0:
        deprocess(noise).save("save/image"+str(i)+".jpg")

hook.remove()
hook2.remove()

### </CORRECTION>

In [ ]:
# --- Convertir l'image en format affichable ---

dream_img = deprocess(noise)
img_orig2  = deprocess(img)
dream_img_c = dream_img.crop((50, 50, 200, 200))
img_orig_c = img_orig2.crop((50, 50, 200, 200))


plt.figure(figsize=(20,5))

plt.subplot(1,4,1)
plt.imshow(img_orig2)
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(dream_img)
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(img_orig_c)
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(dream_img_c)
plt.axis("off")


plt.show()

## F. Transfert de style « à la Gatys » : la matrice de Gram

Références : Gatys, Ecker & Bethge, [*Texture Synthesis Using Convolutional Neural Networks*](https://arxiv.org/abs/1505.07376) (2015) et [*A Neural Algorithm of Artistic Style*](https://arxiv.org/abs/1508.06576) (2015).

### F.0 Qu'est-ce qui ne va pas dans la partie E ?

En E, la loss de style compare les activations **position par position** : $\| A_l(x) - A_l(\text{style}) \|$ avec $A_l \in \mathbb{R}^{C\times H\times W}$.

1. On demande à l'image générée d'avoir, **au même endroit**, les mêmes features que le tableau : on lui impose la *mise en page* du Van Gogh. C'est en réalité une seconde loss de **contenu** (calculée sur l'image de style) : le résultat est une superposition des deux images, pas un chien peint par Van Gogh.
1. Les deux images doivent produire des cartes de même taille $H\times W$. Ça passe ici car `dog.jpg` et `vg.jpg` ont le même format... mais essayez la partie E avec `img/cri.jpg` (format portrait) !

### F.1 Que veut-on capturer avec le « style » ?

Le style d'un peintre, c'est une palette de couleurs, un type de coup de pinceau, des motifs (tourbillons, hachures...). Point clé : **ces éléments sont répartis partout dans le tableau, et leur position n'a pas d'importance**. Un « Van Gogh », ce sont des tourbillons bleus et jaunes, qu'ils soient en haut à gauche ou en bas à droite.

On cherche donc une **statistique des activations qui oublie la position**.

Notons $F \in \mathbb{R}^{C\times N}$ la carte d'activation d'une couche, aplatie ($N = H\times W$ positions, une ligne par canal), et $f_p \in \mathbb{R}^{C}$ sa colonne $p$ : le vecteur des $C$ features qui décrivent l'image autour de la position $p$.

- **Ordre 1** : la moyenne $\mu = \frac1N \sum_p f_p$ dit *quelles* features sont présentes et en quelle quantité... mais pas comment elles se combinent.
- **Ordre 2** : la **matrice de Gram**
$$G = \frac{1}{N}\sum_{p=1}^{N} f_p f_p^\top = \frac1N F F^\top \in \mathbb{R}^{C\times C}, \qquad G_{ij} = \frac1N \sum_p F_{i,p}\,F_{j,p}$$

$G_{ij}$ est grand quand les canaux $i$ et $j$ sont **actifs aux mêmes endroits** : par exemple « trait courbe » ET « jaune » → tourbillon jaune. C'est une matrice de covariance (non centrée) des features.

**Analogie texte** : c'est une matrice de **co-occurrence** de mots : on sait quels mots apparaissent ensemble, mais on a perdu l'ordre des phrases.

**Propriétés** (vérifiées en F.2) :
- $G$ est de taille $C\times C$ **quelle que soit la taille de l'image** → on peut comparer des images de formats différents ;
- $G$ est **invariante à toute permutation des positions** → la mise en page est oubliée ;
- $G$ est symétrique.

Chaque couche apporte sa propre échelle de style : couches basses = couleurs, textures fines ; couches hautes = motifs plus larges et plus complexes. On combine donc **plusieurs couches** $\mathcal{S}$ pour le style, et une couche intermédiaire pour le contenu :
$$\mathcal{L}_{style}(x) = \sum_{l\in \mathcal{S}} \big\| G_l(x) - G_l(\text{style}) \big\|^2 \qquad\qquad \mathcal{L}(x) = \mathcal{L}_{contenu}(x) + \beta\, \mathcal{L}_{style}(x)$$

### F.2 Implémenter la matrice de Gram

Écrire une fonction `gram(feat)` qui prend un tenseur d'activations de forme `(B, C, H, W)` et renvoie les matrices de Gram de forme `(B, C, C)`, normalisées par $N = H\times W$.

*Indices* : `feat.reshape(B, C, H*W)`, puis produit matriciel batché `@` avec `.transpose(1, 2)`.

In [ ]:
def gram(feat):
    b, c, h, w = feat.shape
    f = feat.reshape(b, c, h * w)                 # (B, C, N) : une ligne par canal
    return f @ f.transpose(1, 2) / (h * w)        # (B, C, C)


Exécuter les tests suivants pour vérifier votre implémentation **et** les propriétés annoncées en F.1.

In [ ]:
# --- Test 1 : forme, symétrie, invariance à la permutation des positions ---
torch.manual_seed(0)
A = torch.randn(1, 8, 5, 7)
G = gram(A)
print("forme :", tuple(G.shape))
print("symétrique :", torch.allclose(G, G.transpose(1, 2)))

perm = torch.randperm(5 * 7)                                  # on mélange les pixels de la carte
A_shuffled = A.flatten(2)[:, :, perm].reshape(1, 8, 5, 7)
print("activations modifiées par le mélange :", not torch.allclose(A_shuffled, A))
print("Gram identique après le mélange       :", torch.allclose(gram(A_shuffled), G, atol=1e-6))

# --- Test 2 : la Gram ne dépend pas de la taille de l'image (sur le vrai réseau) ---
activations = {}
h = model.get_submodule("inception3a").register_forward_hook(save_activation("inception3a"))
grams = []
with torch.no_grad():
    for size in [300, 500]:
        model(load_image("img/vg.jpg", size))
        A = activations["inception3a"]
        grams.append(gram(A))
        print(f"image {size}px : activations {tuple(A.shape)} -> Gram {tuple(grams[-1].shape)}")
h.remove()
print("écart relatif entre les deux Gram :", ((grams[0] - grams[1]).norm() / grams[1].norm()).item())

**Question** : combien de coefficients contient la matrice de Gram de `inception3a` ? Et la carte d'activation correspondante pour l'image de 500px ? Qu'en conclure sur ce que « retient » la Gram ?

### <CORRECTION>
**Réponse** : $256 \times 256 = 65\,536$ coefficients (dont ~33 000 indépendants, par symétrie), contre $256 \times 62 \times 79 \approx 1{,}25$ million pour la carte d'activation, et ce nombre ne dépend pas de la taille de l'image. La Gram est un **résumé statistique** : elle garde les co-occurrences de features (le « vocabulaire visuel » du tableau) et jette toute l'information de position (la mise en page). Les deux Gram calculées à 300px et 500px sont proches mais pas identiques : à résolution différente, le réseau ne voit pas exactement les mêmes motifs (un coup de pinceau couvre plus ou moins de pixels).
### </CORRECTION>

### F.3 Synthèse de texture : le style seul

Avant de mélanger contenu et style, vérifions que la Gram capture bien le style : on part d'un **bruit** et on minimise **uniquement** la loss de style. Si la Gram encode bien le « style », on doit obtenir une **texture** à la Van Gogh, sans aucun objet reconnaissable du tableau.

Écrire la fonction `texture_synthesis(style_img, style_layers, nepoch=300, lr=0.05)` qui renvoie l'image générée :
1. poser un hook (fabrique `save_activation` de la partie B) sur chaque couche de `style_layers`
1. calculer les Gram cibles de l'image de style (sans gradient)
1. partir d'un bruit `torch.randn_like(style_img)` et minimiser $\sum_{l} \text{MSE}\big(G_l(x), G_l(\text{style})\big)$ avec Adam + `OneCycleLR`
1. après chaque pas, ramener les pixels dans leur plage valide avec `clamp_pixels(x)` (fournie ci-dessous)
1. **retirer les hooks** à la fin (idéalement dans un `try/finally`, pour qu'ils soient retirés même en cas d'erreur ou d'interruption)

Compter environ 30 secondes sur un Mac M1 pour 300 itérations à 300px (beaucoup plus sur CPU : réduire alors la taille des images).

In [ ]:
# --- Outils fournis ---
# plage valide des pixels après normalisation : (0 - mean)/std ... (1 - mean)/std, par canal
PIX_LOW  = ((0 - torch.tensor(MEAN)) / torch.tensor(STD)).view(1, 3, 1, 1).to(device)
PIX_HIGH = ((1 - torch.tensor(MEAN)) / torch.tensor(STD)).view(1, 3, 1, 1).to(device)

def clamp_pixels(x):
    x.data = torch.clamp(x.data, PIX_LOW, PIX_HIGH)

def show_images(images, titles):
    plt.figure(figsize=(6 * len(images), 5))
    for k, (im, t) in enumerate(zip(images, titles)):
        plt.subplot(1, len(images), k + 1)
        plt.imshow(deprocess(im))
        plt.title(t)
        plt.axis("off")
    plt.show()

style_layers = ["conv2", "inception3a", "inception3b", "inception4a", "inception4c", "inception5a"]

In [ ]:
### <CORRECTION>
def texture_synthesis(style_img, style_layers, nepoch=300, lr=0.05):
    activations.clear()
    handles = [model.get_submodule(n).register_forward_hook(save_activation(n)) for n in style_layers]
    try:
        with torch.no_grad():
            model(style_img)
            target_g = {n: gram(activations[n]) for n in style_layers}

        x = torch.randn_like(style_img).requires_grad_(True)
        optimizer = torch.optim.Adam([x], lr=lr)
        scheduler = OneCycleLR(optimizer, max_lr=lr, total_steps=nepoch)

        for i in range(nepoch):
            optimizer.zero_grad()
            model(x)
            loss = sum(F.mse_loss(gram(activations[n]), target_g[n]) for n in style_layers)
            loss.backward()
            optimizer.step()
            scheduler.step()
            clamp_pixels(x)
            if i % 50 == 0:
                print(f"Iteration {i}, loss style = {loss.item():.4f}")
    finally:
        for h in handles:
            h.remove()
    return x.detach()
### </CORRECTION>

In [ ]:
style = load_image("img/vg.jpg", 300)
texture = texture_synthesis(style, style_layers)
show_images([style, texture], ["style", "texture synthétisée (Gram)"])

### F.4 Transfert de style complet

Écrire la fonction `style_transfer(content_img, style_img, content_layer, style_layers, beta=100, nepoch=300, lr=0.1)` :
- loss de contenu : MSE entre les activations de `content_layer` (pas de Gram ici : pour le contenu, on **veut** garder la mise en page !)
- loss de style : somme des MSE entre matrices de Gram sur `style_layers`
- loss totale : $\mathcal{L}_{contenu} + \beta\,\mathcal{L}_{style}$
- **initialisation** : partir d'une copie de l'image de contenu plutôt que d'un bruit (convergence beaucoup plus rapide)

In [ ]:
### <CORRECTION>
def style_transfer(content_img, style_img, content_layer, style_layers, beta=100, nepoch=300, lr=0.1):
    activations.clear()
    handles = [model.get_submodule(n).register_forward_hook(save_activation(n))
               for n in style_layers + [content_layer]]
    try:
        with torch.no_grad():
            model(content_img)
            target_c = activations[content_layer]
            model(style_img)
            target_g = {n: gram(activations[n]) for n in style_layers}

        x = content_img.clone().requires_grad_(True)   # on part de l'image de contenu
        optimizer = torch.optim.Adam([x], lr=lr)
        scheduler = OneCycleLR(optimizer, max_lr=lr, total_steps=nepoch)

        for i in range(nepoch):
            optimizer.zero_grad()
            model(x)
            loss_c = F.mse_loss(activations[content_layer], target_c)
            loss_s = sum(F.mse_loss(gram(activations[n]), target_g[n]) for n in style_layers)
            loss = loss_c + beta * loss_s
            loss.backward()
            optimizer.step()
            scheduler.step()
            clamp_pixels(x)
            if i % 50 == 0:
                print(f"Iteration {i}, contenu = {loss_c.item():.4f}, style = {loss_s.item():.4f}")
    finally:
        for h in handles:
            h.remove()
    return x.detach()
### </CORRECTION>

In [ ]:
img   = load_image("img/dog.jpg", 400)
style = load_image("img/vg.jpg", 400)
result = style_transfer(img, style, content_layer="inception4b", style_layers=style_layers, beta=100)
show_images([img, style, result], ["contenu", "style", "transfert (Gram)"])

Grâce à la Gram, l'image de style peut avoir un format différent de l'image de contenu :

In [ ]:
style_cri = load_image("img/cri.jpg", 400)     # format portrait : impossible avec la loss de la partie E
result_cri = style_transfer(img, style_cri, content_layer="inception4b", style_layers=style_layers, beta=100)
show_images([img, style_cri, result_cri], ["contenu", "style", "transfert (Gram)"])

### F.5 Expériences et questions

1. Synthétiser une texture avec **uniquement des couches basses** (`["conv2", "inception3a"]`) puis **uniquement des couches hautes** (`["inception4c", "inception5a"]`). Comparer avec la texture de F.3. Que capture chaque groupe de couches ?
1. Faire varier $\beta$ dans le transfert de style (par ex. 0.01, 1, 100). Décrire l'effet.
1. Pourquoi normalise-t-on la Gram par $N = H\times W$ ? Que se passerait-il sans cette normalisation ?
1. Pourquoi n'utilise-t-on **pas** la Gram pour la loss de contenu ?
1. Pourquoi ne pas se contenter de la moyenne des activations par canal (statistique d'ordre 1) ?

In [ ]:
### <CORRECTION>
style = load_image("img/vg.jpg", 300)
tex_low  = texture_synthesis(style, ["conv2", "inception3a"])
tex_high = texture_synthesis(style, ["inception4c", "inception5a"])
show_images([style, tex_low, tex_high, texture],
            ["style", "couches basses", "couches hautes", "toutes les couches (F.3)"])
### </CORRECTION>

In [ ]:
### <CORRECTION>
results = [style_transfer(img, load_image("img/vg.jpg", 400), "inception4b", style_layers, beta=b)
           for b in [0.01, 1, 100]]
show_images(results, [f"beta = {b}" for b in [0.01, 1, 100]])
### </CORRECTION>

### <CORRECTION>
**Réponses**

1. **Couches basses** : la palette de couleurs et le grain fin de la touche (petits traits), mais pas de structure à plus grande échelle. **Couches hautes seules** : l'image reste très bruitée ! Ces couches ont un grand champ réceptif et sont quasi insensibles au détail fin : rien ne contraint les pixels à petite échelle, le bruit initial n'est donc jamais « nettoyé ». Avec **toutes les couches**, chaque échelle est contrainte : couleurs et touche (couches basses), organisation des traits en tourbillons plus larges (couches intermédiaires/hautes). C'est pourquoi on somme sur plusieurs couches.
1. $\beta = 0.01$ : l'image reste identique au contenu. $\beta = 1$ : la touche de Van Gogh apparaît nettement. $\beta = 100$ : couleurs et coups de pinceau envahissent l'image, mais le chien reste reconnaissable. $\beta$ règle le compromis contenu/style ; au-delà d'une certaine valeur l'effet sature, car on part de l'image de contenu et Adam normalise l'amplitude du gradient : seul le *rapport* entre les deux loss compte, et le nombre d'itérations limite la dérive.
1. Sans normalisation, $G = F F^\top$ est une **somme** sur les positions : sa valeur grandit avec la taille de l'image (on ne pourrait plus comparer un style à 300px avec une image à 400px), et les couches basses (cartes spatialement grandes) écraseraient les couches hautes dans la loss. Avec la normalisation, $G$ est une **moyenne** : une statistique indépendante de la taille. (La normalisation par $C\times H\times W$, que l'on trouve aussi souvent, rend en revanche les couches profondes, à grand $C$, quasiment invisibles dans la somme : il faut alors les repondérer.)
1. Le contenu, c'est justement *ce qui est où* : la position du chien, de l'herbe, de la haie. La Gram jetterait cette information : on obtiendrait une texture « de chien », pas le chien.
1. La moyenne dit quelles features sont présentes, pas lesquelles apparaissent **ensemble**. Ex : « bleu » et « courbe » peuvent être présents tous les deux dans deux images, mais seule la Gram distingue des courbes bleues de courbes jaunes sur fond bleu. (Des méthodes plus récentes, comme AdaIN de Huang & Belongie 2017, utilisent moyenne + écart-type par canal : moins riche, mais bien plus rapide.)
### </CORRECTION>

# Construction du sujet à partir de la correction

In [54]:
### <CORRECTION> ###
import re
# transformation de cet énoncé en version étudiante

fname = "2_4-CNN-deepdreams-corr.ipynb" # ce fichier
fout  = fname.replace("-corr","")

# print("Fichier de sortie: ", fout )

f = open(fname, "r")
txt = f.read()

f.close()


f2 = open(fout, "w")
f2.write(re.sub("<CORRECTION>.*?(</CORRECTION>)"," TODO ",\
    txt, flags=re.DOTALL))
f2.close()

### </CORRECTION> ###